In [1]:
# Import libries
import numpy as np
import pandas as pd
import os
import re
import glob
import json
import csv
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyranges as pr
import pysam
# from scipy.stats import mannwhitneyu, stats
# from statannotations.Annotator import Annotator

current_directory = os.getcwd()
print("Current Directory:", current_directory)
pd.set_option("display.max_columns", None)


Current Directory: /mnt/NAS3/home/jiwon/ECTRES/python


In [2]:
# find -L /mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/results/align_dna/GRCh37 -name "*.bam" -exec realpath {} + > /mnt/NAS3/home/jiwon/ECTRES/data/bam_list_all_origin.txt

In [3]:
# 파일 경로 설정
file_path = '/mnt/NAS3/home/jiwon/ECTRES/data/bam_list_all_origin.txt'

# 파일 읽기
with open(file_path, 'r') as f:
    bam_list = f.read().splitlines()

# 결과 확인 (개수와 상위 5개 출력)
print(f"총 {len(bam_list)}개의 BAM 파일이 로드되었습니다.")
print("상위 5개 파일:")
for path in bam_list[:5]:
    print(path)

총 77개의 BAM 파일이 로드되었습니다.
상위 5개 파일:
/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/results/align_dna/GRCh37/ECTRES-ECGI1-0001-TPX-A01-WGS-1ST985/applyBqsr/ECTRES-ECGI1-0001-TPX-A01-WGS-1ST985.realn.mdup.bqsr.bam
/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/results/align_dna/GRCh37/ECTRES-ECGI1-0001-TPX-A01-WGS-6DM349/applyBqsr/ECTRES-ECGI1-0001-TPX-A01-WGS-6DM349.realn.mdup.bqsr.bam
/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/results/align_dna/GRCh37/ECTRES-ECGI1-0001-TPX-A02-WGS-5KM079/applyBqsr/ECTRES-ECGI1-0001-TPX-A02-WGS-5KM079.realn.mdup.bqsr.bam
/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/results/align_dna/GRCh37/ECTRES-ECGI1-0001-TPX-A03-WGS-0LT586/applyBqsr/ECTRES-ECGI1-0001-TPX-A03-WGS-0LT586.realn.mdup.bqsr.bam
/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/results/align_dna/GRCh37/ECTRES-ECGI1-0001-TPX-A04-WGS-0SR571/applyBqsr/ECTRES-ECGI1-0001-TPX-A04-WGS-0SR571.realn.mdup.bqsr.bam


In [4]:
manifest=pd.read_csv('../manifest/ECTRES_clones_nf_dna_fastqs_20260303.csv')
manifest.head(2)

manifest["sample_id"] = manifest["sample_legacy_id"].fillna("parental")
sample_mapping = manifest[['aliquot_barcode','source_barcode','sample_barcode','patient_barcode','sample_id']].drop_duplicates()

print(manifest.shape, sample_mapping.shape)

(87, 19) (77, 5)


In [5]:
sample_mapping.groupby('source_barcode').size()

source_barcode
ECGI1    34
EFM19    11
H2170    32
dtype: int64

In [6]:
sample_mapping.head()

,aliquot_barcode,source_barcode,sample_barcode,patient_barcode,sample_id
0,ECTRES-ECGI1-0001-TPX-A01-WGS-6DM349,ECGI1,ECTRES-ECGI1-0001-TPX-A01,ECTRES-ECGI1-0001,EG_1
2,ECTRES-ECGI1-0001-TPX-A10-WGS-3SW949,ECGI1,ECTRES-ECGI1-0001-TPX-A10,ECTRES-ECGI1-0001,EG_10
3,ECTRES-ECGI1-0001-TPX-A11-WGS-9HJ669,ECGI1,ECTRES-ECGI1-0001-TPX-A11,ECTRES-ECGI1-0001,EG_11
4,ECTRES-ECGI1-0001-TPX-A12-WGS-4SL389,ECGI1,ECTRES-ECGI1-0001-TPX-A12,ECTRES-ECGI1-0001,EG_12
5,ECTRES-ECGI1-0001-TPX-A13-WGS-3VZ640,ECGI1,ECTRES-ECGI1-0001-TPX-A13,ECTRES-ECGI1-0001,EG_13
